# Get YouTube Subtitles

**Supplementary code for:** *[Anonymous Author]. Turkish Child Lexicon (TCLex): A lexical database of various media for Turkish primary school children. Behavior Research Methods.*


This notebook provides a small toolkit built on the [YouTube Data API v3](https://developers.google.com/youtube/v3) and the
[`youtube_transcript_api`](https://github.com/jdepoix/youtube-transcript-api) package for collecting subtitle (caption) data from
YouTube videos. It was used to download subtitles for the video corpus analyzed in the article above. Given a YouTube **channel ID**
or **playlist ID**, the code retrieves the associated video IDs and then downloads the available subtitles (in `.srt` format) for
each video in a chosen language.

## Contents

1. Setup (dependencies, imports, API key)
2. Core functions
   - `get_video_ids_from_channel` — list video IDs uploaded to a channel
   - `get_video_ids_from_playlist` — list video IDs in a playlist
   - `get_playlists` — list all playlists on a channel
   - `get_subtitles` — download subtitles for a list of video IDs
   - `count_items_in_folder` — count files saved to an output folder
3. Example usage
4. Alternative usage patterns
5. Listing a channel's playlists
6. Utility: counting downloaded subtitle files

## Requirements

- A YouTube Data API v3 key (see the *API key* section below for how to obtain one and its free daily quota).
- Python packages: `requests`, `youtube_transcript_api`.
- Not every video has subtitles in every language; `get_subtitles` silently skips videos for which no transcript is available in
  the requested language and prints a short notice instead of raising an error, so a full run does not stop partway through.

## License

MIT License — see LICENSE file.


#Setup

### 1.1 Install dependencies


In [ ]:
!pip install youtube_transcript_api --quiet

In [ ]:
import os
import glob

import requests
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api.formatters import SRTFormatter

##1.2 Get YouTube API
`get_video_ids_from_channel`, `get_video_ids_from_playlist`, and `get_playlists` call the YouTube Data API v3, which requires a
free API key tied to a Google Cloud project (see the
[getting started guide](https://developers.google.com/youtube/v3/getting-started)).

`get_subtitles` does **not** use this key — it calls `youtube_transcript_api`, which reads the same public captions a viewer sees
on youtube.com and needs no credentials.


In [ ]:
#YouTube API key to get video_ids
api_key = "YOUR_API_KEY_HERE"

##1.3 (Optional) Mount Drive
This notebook was originally created and run in Google Colab, saving outputs directly to Google Drive. If you wish to use Google Drive as your output, mount Google Drive. Otherwise, you can skip this.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("Not running in Google Colab — skipping Drive mount. "
          "Set output paths below to a local folder instead.")

# 2. Functions to get Video IDs and subtitles

The five functions below cover the full pipeline: discover video IDs (from a channel or a playlist), optionally discover
playlists on a channel, download subtitles for a set of video IDs, and count the files produced.

In [ ]:
import os
import requests
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api.formatters import SRTFormatter

def get_video_ids_from_channel(api_key, channel_id):
  """Function to retrieve video_ids from a channel

  Args:
        api_key (str): YouTube API key.
        channel_id (str): YouTube Channel ID.

    Returns:
        list: A list of video IDs.

  """

  video_ids = []
  base_url = "https://www.googleapis.com/youtube/v3/search"
  params = {
      "part": "id",
      "channelId": channel_id,
      "maxResults": 50,
      "order": "date",
      "type": "video",
      "key": api_key
  }

  while True:
      response = requests.get(base_url, params=params)
      data = response.json()

      for item in data.get("items", []):
          video_ids.append(item["id"]["videoId"])

      if "nextPageToken" in data:
          params["pageToken"] = data["nextPageToken"]
      else:
          break

  return video_ids



def get_video_ids_from_playlist(api_key, playlist_id):
  """
  Function to retrieve video IDs from a specific YouTube playlist.

  Args:
      api_key (str): YouTube API key.
      playlist_id (str): YouTube Playlist ID.

  Returns:
      list: A list of video IDs from the playlist.
  """
  video_ids = []
  base_url = "https://www.googleapis.com/youtube/v3/playlistItems"
  params = {
      "part": "contentDetails",
      "playlistId": playlist_id,
      "maxResults": 50,
      "key": api_key
  }

  while True:
      response = requests.get(base_url, params=params)
      data = response.json()

      for item in data.get("items", []):
          video_ids.append(item["contentDetails"]["videoId"])

      if "nextPageToken" in data:
          params["pageToken"] = data["nextPageToken"]
      else:
          break

  return video_ids



def get_subtitles(video_ids,lang,output_dir):
  """Fetch captions for given video IDs and write to files.

    Args:
      video_ids (list): YouTube video ids.
      lang (str): Language code (en, tr).
      output_dir (str): Directory to save the captions.

  Returns:
      None


  """


  if not os.path.exists(output_dir):
      os.makedirs(output_dir)

  for video_id in video_ids:
      try:
          transcript = YouTubeTranscriptApi.get_transcript(video_id, languages=[lang])

          formatter = SRTFormatter()
          srt_captions = formatter.format_transcript(transcript)

          with open(f'{output_dir}/{video_id}.srt', 'w', encoding='utf-8') as f:
              f.write(srt_captions)

          print(f"Captions for video {video_id} saved to 'captions/{video_id}.srt'")

      except Exception as e:
          print(f"Could not retrive captions for {video_id}'")


def get_playlists(api_key, channel_id):
    """
    Function to retrieve all playlist IDs and names for a YouTube channel.

    Args:
        api_key (str): YouTube API key.
        channel_id (str): YouTube Channel ID.

    Returns:
        list: A list of dictionaries, where each dictionary contains 'playlist_id' and 'playlist_name'.
    """
    playlists = []
    base_url = "https://www.googleapis.com/youtube/v3/playlists"
    params = {
        "part": "snippet",
        "channelId": channel_id,
        "maxResults": 50,
        "key": api_key
    }

    while True:
        response = requests.get(base_url, params=params)
        data = response.json()

        # Log errors if any
        if "error" in data:
            raise Exception(f"API Error: {data['error']['message']}")

        # Extract playlist IDs and names
        if "items" in data:
            for item in data["items"]:
                playlist_id = item["id"]
                playlist_name = item["snippet"]["title"]
                playlists.append({"playlist_id": playlist_id, "playlist_name": playlist_name})

        # Break if no nextPageToken
        if "nextPageToken" in data:
            params["pageToken"] = data["nextPageToken"]
        else:
            break

    return playlists

def count_items_in_folder(folder_path):
    items = glob.glob(os.path.join(folder_path, '**', '*'), recursive=True)

    return len(items)

#3. Example Usage

## 3.1. Get video ids from playlist

The following dictionary contains some playlists each of which contains various videos. To get the video_ids, run the code below with your own YouTube playlists.




In [ ]:
# A simpler example with only one playlist
PLAYLIST_ID = 'PLcycGAI5cENRXEuinMJU64x4YtPgM88Nr'
video_ids_1 = get_video_ids_from_playlist(api_key,PLAYLIST_ID)


In [ ]:
#A more complex example with many playlists.
#Playlists are first listed in a dictionary so that each video id is
# associated with a playlist.
playlists = {"tozkoparan_iskender":"PL_VIYA-L9VnL70LnyRdusA6b3ZdY1FhSR",
              "cocuk_bahcesi":"PL_VIYA-L9VnKj42CAhFTNqJd76_eGcc8t",
              "kuzucuk":"PL_VIYA-L9VnKje-wVqwbyUx1hvI7b7dtb",
              "ciciki":"PL_VIYA-L9VnKZQqrcHqGbBqRsYA9hyLbE",
              "canım_kardesim":"PL_VIYA-L9VnITCPplS0uzKgyjJucmeihO",
              "keloglan":"PL_VIYA-L9VnJOq1reafpJDzFeqmPSZZLI"
            }


video_ids_dict = {}

for playlist_name, playlist_id in playlists.items():
    video_ids_dict[playlist_name] = get_video_ids_from_playlist(api_key, playlist_id)



## 3.2 Get video ids from channel

The following code illustrates getting video ids from a channel.

In [ ]:
CHANNEL_ID = "UCxRX_QTK_L-Vwon_H2POnfg"
video_ids = get_video_ids_from_channel(api_key,CHANNEL_ID)

## 3.3 Get playlists from channel

The following code illustrates getting playlists from a channel.

In [ ]:
channel_id ="UCxRX_QTK_L-Vwon_H2POnfg" # sp_cocuk
playlists = get_playlists(api_key,channel_id)

##3.4 Get subtitles from video ids.

In [ ]:
output_path = "YOUR_OUTPUT_FOLDER"
lang = "tr"

get_subtitles(video_ids,lang,output_path)

#4 Count downloaded Subtitle Files
The code below counts the number of subtitle files in a given folder.

In [ ]:
folder_path = 'YOUR_OUTPUT_FOLDER'
subtitle_count = count_items_in_folder(folder_path)

print(f"{subtitle_count} subtitles")